# Phase 4: Supervised Learning - V2 Architecture (Stacking Ensemble)

In this notebook, we implement the production-ready **Version 2 (V2)** of our machine learning architecture. 

**Architectural Upgrades:**
1. **Unified Preprocessing Pipeline:** A robust `ColumnTransformer` handling `StandardScaler` for numericals and `OneHotEncoder` for categoricals, preventing data leakage.
2. **Stacking Ensemble Classifier:** Instead of relying on a single algorithm, we combine the predictive power of `RandomForest`, `SVC`, and `GradientBoosting` as base learners, utilizing `LogisticRegression` as the meta-learner to output the final calibrated probability.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, accuracy_score

# Plot settings
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context("paper", font_scale=1.2)

## 1. Data Loading & Stratified Splitting
We load the raw data and split it into training and testing sets. Stratification ensures the class ratio is maintained across both sets.

In [ ]:
# Load data
df = pd.read_csv('../data/raw/heart.csv')
X = df.drop(columns=['HeartDisease'])
y = df['HeartDisease']

# Stratified Train-Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples")

## 2. The Unified Preprocessing Pipeline
This is the exact preprocessing logic used in `src/heart_disease/features.py`. It acts as the single source of truth for data transformations.

In [ ]:
# Define feature groups
numeric_features = ['Age', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR', 'Oldpeak']
categorical_features = ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']

# Numeric Pipeline
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical Pipeline
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine into ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

## 3. Defining the Stacking Ensemble
We define three diverse base estimators. Their predictions are fed into a Logistic Regression meta-learner via 5-fold cross-validation to prevent overfitting.

In [ ]:
# Define Base Learners
base_estimators = [
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('svc', SVC(probability=True, random_state=42)),
    ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42))
]

# Define Meta-Learner
meta_learner = LogisticRegression()

# Construct Stacking Classifier
stacking_clf = StackingClassifier(
    estimators=base_estimators,
    final_estimator=meta_learner,
    cv=5
)

# Final End-to-End Pipeline
final_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', stacking_clf)
])

## 4. Model Training & Standard Evaluation
We train the entire pipeline on the training set and evaluate it on the unseen test set.

In [ ]:
# Train the pipeline
print("Training Stacking Ensemble Pipeline...")
final_pipeline.fit(X_train, y_train)

# Predict
y_pred = final_pipeline.predict(X_test)

# Evaluate
print(f"\nFinal Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

## 5. Advanced Evaluation: Confusion Matrix
Visualizing the confusion matrix helps us understand the False Positives and False Negatives, which are critical in clinical diagnostics.

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, 
            xticklabels=['Normal (0)', 'Disease (1)'], 
            yticklabels=['Normal (0)', 'Disease (1)'])
plt.title('Stacking Ensemble - Confusion Matrix', weight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

## 6. Advanced Evaluation: ROC Curve Comparison
To prove the superiority of the Stacking Ensemble, we plot its Receiver Operating Characteristic (ROC) curve against the individual base learners.

In [ ]:
plt.figure(figsize=(10, 8))

# 1. Evaluate Base Learners individually
for name, estimator in base_estimators:
    # Create a temporary pipeline for each base learner
    temp_pipe = Pipeline(steps=[('preprocessor', preprocessor), ('clf', estimator)])
    temp_pipe.fit(X_train, y_train)
    
    # Get probabilities
    y_proba = temp_pipe.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)
    
    plt.plot(fpr, tpr, lw=2, alpha=0.6, label=f'{name.upper()} (AUC = {roc_auc:.3f})')

# 2. Evaluate Stacking Ensemble
y_proba_stack = final_pipeline.predict_proba(X_test)[:, 1]
fpr_stack, tpr_stack, _ = roc_curve(y_test, y_proba_stack)
roc_auc_stack = auc(fpr_stack, tpr_stack)

plt.plot(fpr_stack, tpr_stack, color='darkred', lw=3, label=f'Stacking Ensemble (AUC = {roc_auc_stack:.3f})')

# Plot formatting
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', weight='bold')
plt.ylabel('True Positive Rate', weight='bold')
plt.title('ROC Curve Comparison: Base Learners vs. Stacking Ensemble', weight='bold')
plt.legend(loc="lower right")
plt.show()